In [ ]:
# heat map used in the manuscript
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load data
df = pd.read_csv('upregulate_heapmap-reranked-4-2.csv')

# Define columns
dmso_cols = ['DMSO_1', 'DMSO_2', 'DMSO_3']
lle_cols = ['LEE_1', 'LEE_2', 'LEE_3']
la_cols  = ['LA_1',  'LA_2',  'LA_3']
all_sample_cols = dmso_cols + lle_cols + la_cols

# Index by gene symbol and select expression matrix
df = df.set_index('symbol')
exp_data = df[all_sample_cols]

# Compute DMSO-centered z-scores
dmso_mean = df[dmso_cols].mean(axis=1)
row_std = exp_data.std(axis=1)

safe_std = row_std.replace(0, np.nan)
exp_data_zscore = exp_data.sub(dmso_mean, axis=0).div(safe_std, axis=0)
exp_data_zscore = exp_data_zscore.fillna(0.0)

# Clip for visualization
exp_data_zscore = exp_data_zscore.clip(-3, 3)

# Set font globally to Arial
plt.rcParams['font.family'] = 'Arial'

# Plot
plt.figure(figsize=(5, 8))
im = plt.imshow(
    exp_data_zscore.values,
    aspect='auto',
    cmap='coolwarm',
    vmin=-3,
    vmax=3  
)

plt.xticks(np.arange(len(all_sample_cols)), all_sample_cols, rotation=45, ha='right')

# Highlight certain genes
highlight_genes = {"GPNMB", "PGLYRP2", "CCDC141", "PDE3A"}
plt.yticks(np.arange(len(exp_data)), exp_data.index)

ax = plt.gca()
for label in ax.get_yticklabels():
    gene = label.get_text()
    if gene in highlight_genes:
        label.set_fontweight('bold')
        label.set_fontsize(9)
    else:
        label.set_fontsize(9)
        label.set_color('black')

# Colorbar with clearer label
cbar = plt.colorbar(im, shrink=0.4)
cbar.set_label('Z-score (relative to DMSO)')

plt.title('Gene Expression (Z-score, DMSO-centered)', loc='left')
plt.tight_layout()
#plt.savefig('RNA-heatmap_plasma-highlighted-2.png', dpi=300)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('VolcanoCalls_LEE_vs_DMSO.csv')

# Color map for categories
color_dict = {'Up': '#B40426', 'Down': '#3B4CC0', 'NS': '#A8A8A8'}
df['color'] = df['Sig'].map(color_dict)

# –log10(padj), avoid log of zero
df['-log10(padj)'] = -np.log10(df['padj'].replace(0, np.nan))


plt.figure(figsize=(7, 6))
plt.rcParams['font.family'] = 'Arial'

plt.grid(which='major', axis='both', color='#DDDDDD', linewidth=1.0)
plt.gca().set_axisbelow(True)

plt.minorticks_on()
plt.grid(which='minor', axis='both', color='#EEEEEE', linewidth=0.5, linestyle=':')

# Axis ranges
x_min, x_max = -4, 4
y_min, y_max = 0, 200

plt.yticks(range(0, 201, 50))

# Filter points within axis limits (hide outliers)
df_filtered = df[(df['log2FoldChange'].between(x_min, x_max)) &
                 (df['-log10(padj)'].between(y_min, y_max))]

# Plot each group in desired legend order
for status in ['Down', 'NS', 'Up']:
    subset = df_filtered[df_filtered['Sig'] == status]
    plt.scatter(
        subset['log2FoldChange'],
        subset['-log10(padj)'],
        color=subset['color'],
        label=status,
        s=20 if status != 'NS' else 20,
        alpha=0.8 if status != 'NS' else 0.5,
        edgecolor='none'
    )

# Annotate top genes (within visible range only)
for status, text_color in [('Up', '#B40426'), ('Down', '#3B4CC0')]:
    top = df_filtered[df_filtered['Sig'] == status].sort_values('padj').head(7)
    for _, row in top.iterrows():
        plt.text(row['log2FoldChange']+ 0.1, row['-log10(padj)']+ 0.05, row['symbol'],
                 color=text_color, fontsize=10, family='Arial')
#coolwarm, blu end #3B4CC0, red end #B40426
# classic ('Up', '#EF3B2C'), ('Down', '#2C66B8')

# Threshold lines
log2FC_thr = 1
padj_thr = 0.05
plt.axvline(-log2FC_thr, color='k', linestyle=':', lw=1)
plt.axvline(log2FC_thr, color='k', linestyle=':', lw=1)
plt.axhline(-np.log10(padj_thr), color='k', linestyle='--', lw=1)

# Set limits
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max * 1.1)  # add 10% headroom for text/labels

# Labels & title
plt.xlabel('log2 fold-change', fontsize=12, family='Arial')
plt.ylabel('-log10(FDR)', fontsize=12, family='Arial')
plt.title('LEE vs DMSO', loc='left', fontsize=16, family='Arial', weight='bold')

# Add descriptive text above plot area
plt.text(x_min, y_max * 1.04,
         " | padj < 0.05, |log2FC| > =1 | x-range ± 4, y-range 0–200",
         fontsize=12, family='Arial')

# Legend
plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, 1.07),
           ncol=3, frameon=False)

# Remove the black square frame
ax = plt.gca()
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(False)

#ax = plt.gca()
#ax.spines['bottom'].set_visible(False)
#ax.spines['left'].set_visible(False)

plt.tight_layout()
#plt.savefig('VolcanoCalls_LEE_vs_DMSO_coolwarm-2.png', dpi=300)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('VolcanoCalls_LA_vs_DMSO.csv')

# Color map for categories
color_dict = {'Up': '#B40426', 'Down': '#2C66B8', 'NS': '#A8A8A8'}
df['color'] = df['Sig'].map(color_dict)

# –log10(padj), avoid log of zero
df['-log10(padj)'] = -np.log10(df['padj'].replace(0, np.nan))

#B40426
plt.figure(figsize=(7, 6))
plt.rcParams['font.family'] = 'Arial'

plt.grid(which='major', axis='both', color='#DDDDDD', linewidth=1.0)
plt.gca().set_axisbelow(True)

plt.minorticks_on()
plt.grid(which='minor', axis='both', color='#EEEEEE', linewidth=0.5, linestyle=':')

# Axis ranges
x_min, x_max = -4, 4
y_min, y_max = 0, 200

plt.yticks(range(0, 201, 50))

# Filter points within axis limits (hide outliers)
df_filtered = df[(df['log2FoldChange'].between(x_min, x_max)) &
                 (df['-log10(padj)'].between(y_min, y_max))]

# Plot each group in desired legend order
for status in ['Down', 'NS', 'Up']:
    subset = df_filtered[df_filtered['Sig'] == status]
    plt.scatter(
        subset['log2FoldChange'],
        subset['-log10(padj)'],
        color=subset['color'],
        label=status,
        s=20 if status != 'NS' else 20,
        alpha=0.8 if status != 'NS' else 0.5,
        edgecolor='none'
    )

# Annotate top genes (within visible range only)
for status, text_color in [('Up', '#B40426'), ('Down', '#3B4CC0')]:
    top = df_filtered[df_filtered['Sig'] == status].sort_values('padj').head(7)
    for _, row in top.iterrows():
        plt.text(row['log2FoldChange']+ 0.1, row['-log10(padj)']+ 0.05, row['symbol'],
                 color=text_color, fontsize=10, family='Arial')
#coolwarm, blu end #3B4CC0, red end #B40426
# classic ('Up', '#EF3B2C'), ('Down', '#2C66B8')

# Threshold lines
log2FC_thr = 1
padj_thr = 0.05
plt.axvline(-log2FC_thr, color='k', linestyle=':', lw=1)
plt.axvline(log2FC_thr, color='k', linestyle=':', lw=1)
plt.axhline(-np.log10(padj_thr), color='k', linestyle='--', lw=1)

# Set limits
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max * 1.1)  # add 10% headroom for text/labels

# Labels & title
plt.xlabel('log2 fold-change', fontsize=12, family='Arial')
plt.ylabel('-log10(FDR)', fontsize=12, family='Arial')
plt.title('LA vs DMSO', loc='left', fontsize=16, family='Arial', weight='bold')

# Add descriptive text above plot area
plt.text(x_min, y_max * 1.04,
         " | padj < 0.05, |log2FC| > =1 | x-range ± 4, y-range 0–200",
         fontsize=12, family='Arial')

# Legend
plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, 1.07),
           ncol=3, frameon=False)

# Remove the black square frame
ax = plt.gca()
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(False)

#ax = plt.gca()
#ax.spines['bottom'].set_visible(False)
#ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.savefig('VolcanoCalls_LA_vs_DMSO_coolwarm-2.png', dpi=300)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# Load data
df = pd.read_csv('VolcanoCalls_LA_vs_LEE_EQUIPOTENT_1xIC50.csv')

# Colors (coolwarm ends for Up/Down)
color_dict = {'Up': '#B40426', 'Down': '#3B4CC0', 'NS': '#A8A8A8'}
df['color'] = df['Sig'].map(color_dict)

# -log10(padj)
df['-log10(padj)'] = -np.log10(df['padj'].replace(0, np.nan))

plt.figure(figsize=(7, 6)) # a bit wider helps placement
plt.rcParams['font.family'] = 'Arial'
ax = plt.gca()
ax.set_axisbelow(True)
plt.grid(which='major', axis='both', color='#DDDDDD', linewidth=1.0)
plt.minorticks_on()
plt.grid(which='minor', axis='both', color='#EEEEEE', linewidth=0.5, linestyle=':')

# Axis ranges
x_min, x_max = -4, 4
y_min, y_max = 0, 200
plt.yticks(range(0, 201, 50))

# Keep only visible points
vis = df[(df['log2FoldChange'].between(x_min, x_max)) &
         (df['-log10(padj)'].between(y_min, y_max))]

# Scatter (keep references to pass into adjust_text via add_objects)
scatters = []
for status in ['Down', 'NS', 'Up']:
    sub = vis[vis['Sig'] == status]
    sc = plt.scatter(
        sub['log2FoldChange'], sub['-log10(padj)'],
        color=sub['color'], s=22,
        alpha=0.85 if status != 'NS' else 0.5,
        edgecolor='none', label=status
    )
    scatters.append(sc)

# Helper to set initial alignment and offset
def _ha(x): return 'right' if x < 0 else 'left'
def _dx(x): return -0.18 if x < 0 else 0.18  # push labels outward from center

# Collect annotations with initial offsets & category-colored connectors
texts = []

# Up (red) — stagger initial y-text to reduce starting collisions
top_up = vis[vis['Sig'] == 'Up'].sort_values('padj').head(7).reset_index(drop=True)
for i, r in top_up.iterrows():
    x, y = r['log2FoldChange'], r['-log10(padj)']
    ann = plt.annotate(
        r['symbol'],
        xy=(x, y),
        xytext=(x + _dx(x)-0.3 , y + 7 + i * 3),  # outward + upward stagger
        textcoords='data',
        color='#B40426', fontsize=10, family='Arial',
        ha=_ha(x), va='bottom', clip_on=False,
        arrowprops=dict(arrowstyle='-', lw=0.9, color='#B40426', alpha=0.85)
    )
    texts.append(ann)
#light red #EF3B2C; lighter blue (#2C66B8).
# Down (blue)
top_down = vis[vis['Sig'] == 'Down'].sort_values('padj').head(7).reset_index(drop=True)
for i, r in top_down.iterrows():
    x, y = r['log2FoldChange'], r['-log10(padj)']
    ann = plt.annotate(
        r['symbol'],
        xy=(x, y),
        xytext=(x + _dx(x), y + 5 + i * 3),
        textcoords='data',
        color='#3B4CC0', fontsize=10, family='Arial',
        ha=_ha(x), va='bottom', clip_on=False,
        arrowprops=dict(arrowstyle='-', lw=0.9, color='#3B4CC0', alpha=0.85)
    )
    texts.append(ann)

# ---- Auto-adjust (stronger forces, more iters, avoid points) ----
adjust_text(
    texts,
    add_objects=scatters,                     # avoid overlapping scatter points
    autoalign='x',                            # prefer vertical moves
    only_move={'points':'', 'text':'xy'},     # don't move points; move text in x/y
    expand_points=(1.4, 1.6),
    expand_text=(1.3, 1.4),
    force_points=(0.2, 0.5),                  # repulsion strength against points
    force_text=(0.1, 0.8),                   # repulsion strength among labels
    lim=1200,                                 # more iterations if needed
    precision=0.001
)

# Threshold lines
log2FC_thr = 1
padj_thr = 0.05
plt.axvline(-log2FC_thr, color='k', linestyle=':', lw=1)
plt.axvline(log2FC_thr,  color='k', linestyle=':', lw=1)
plt.axhline(-np.log10(padj_thr), color='k', linestyle='--', lw=1)

# Limits + extra headroom for labels
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max * 1.1)  # a bit more top room for moved labels

# Labels & title
plt.xlabel('log2 fold-change', fontsize=12)
plt.ylabel('-log10(FDR)', fontsize=12)
plt.title('LA vs LEE', loc='left', fontsize=16, weight='bold')

# Header text
plt.text(x_min, y_max * 1.04,
         " | padj < 0.05, |log2FC| ≥ 1 | x-range ±4, y-range 0–200",
         fontsize=12)

# Legend
plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.07), ncol=3, frameon=False)

# Frameless look
for side in ['top', 'right', 'left', 'bottom']:
    ax.spines[side].set_visible(False)

plt.tight_layout()
#plt.savefig('VolcanoCalls_LA_vs_LEE_equipotent_coolwarm-auto-placed-5.png', dpi=300)
plt.show()
